# RQ3 statistical test

Original notebook used for statistical comparison of saved RQ3 results.


## Path configuration

The analysis logic below is unchanged from the research notebook. In the
configuration cell, replace every `REPLACE_WITH_...` value with the path to
the named file or directory on your machine. Paths may be absolute, or
relative to the repository root when Jupyter is started from that directory.
Generated outputs are written under the repository's `outputs/` directory.


# RQ3 Statistical Significance — Paired Two-Tailed t-test




- NCDM-poly
- GLLMCDM w/o attempt
- GLLMCDM





|---|---|---|




\[
d_r=MAE_{candidate,r}-MAE_{G\text{-}NCDM,r}
\]

\[
H_0:\mu_d=0,\qquad H_1:\mu_d\ne 0
\]



In [ ]:
# CELL A1 — imports and project paths
from pathlib import Path
from itertools import combinations
import math

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd().resolve()
NOTEBOOK_DIR = PROJECT_ROOT / "outputs" / "RQ3_TTest"

SETTINGS = ["S1", "S2", "S3"]
TARGETS = ["Mid", "Final", "Avg"]
REFERENCE_MODEL = "G-NCDM-poly"
CANDIDATE_MODELS = ["NCDM-poly", "GLLMCDM w/o attempt", "GLLMCDM"]
EXPECTED_ROUNDS = set(range(1, 31))
ALPHA = 0.05

print("Project root:", PROJECT_ROOT)
print("Reference model:", REFERENCE_MODEL)
print("Expected rounds: 1-30; split code uses random_state = 42 + round")


In [ ]:
# CELL A2 — source folders for all 12 model × setting runs
# Each folder must contain predictions.csv, per_round.csv, and summary.csv.
SOURCE_DIRS = {
    ("GLLMCDM", "S1"): Path("REPLACE_WITH_GLLMCDM_S1_RQ3_OUTPUT_DIR"),
    ("GLLMCDM", "S2"): Path("REPLACE_WITH_GLLMCDM_S2_RQ3_OUTPUT_DIR"),
    ("GLLMCDM", "S3"): Path("REPLACE_WITH_GLLMCDM_S3_RQ3_OUTPUT_DIR"),
    ("GLLMCDM w/o attempt", "S1"): Path("REPLACE_WITH_GLLMCDM_WO_ATTEMPT_S1_RQ3_OUTPUT_DIR"),
    ("GLLMCDM w/o attempt", "S2"): Path("REPLACE_WITH_GLLMCDM_WO_ATTEMPT_S2_RQ3_OUTPUT_DIR"),
    ("GLLMCDM w/o attempt", "S3"): Path("REPLACE_WITH_GLLMCDM_WO_ATTEMPT_S3_RQ3_OUTPUT_DIR"),
    ("G-NCDM-poly", "S1"): Path("REPLACE_WITH_GNCDM_POLY_S1_RQ3_OUTPUT_DIR"),
    ("G-NCDM-poly", "S2"): Path("REPLACE_WITH_GNCDM_POLY_S2_RQ3_OUTPUT_DIR"),
    ("G-NCDM-poly", "S3"): Path("REPLACE_WITH_GNCDM_POLY_S3_RQ3_OUTPUT_DIR"),
    ("NCDM-poly", "S1"): Path("REPLACE_WITH_NCDM_POLY_S1_RQ3_OUTPUT_DIR"),
    ("NCDM-poly", "S2"): Path("REPLACE_WITH_NCDM_POLY_S2_RQ3_OUTPUT_DIR"),
    ("NCDM-poly", "S3"): Path("REPLACE_WITH_NCDM_POLY_S3_RQ3_OUTPUT_DIR"),
}

source_manifest = pd.DataFrame([
    {
        "setting": setting,
        "model": model,
        "source_dir": str(path),
        "exists": path.is_dir(),
    }
    for (model, setting), path in SOURCE_DIRS.items()
]).sort_values(["setting", "model"]).reset_index(drop=True)

display(source_manifest)
if len(SOURCE_DIRS) != 12 or not source_manifest["exists"].all():
    raise FileNotFoundError("Expected 12 source folders and all must exist.")


In [ ]:
# CELL A3 — validation helpers and Holm adjustment
PRED_REQUIRED = {
    "setting", "model_name", "target_exam", "round",
    "user_id", "item_id", "y_true", "y_pred", "abs_error",
}
ROUND_REQUIRED = {
    "setting", "model_name", "target_exam", "round", "mae",
}
SUMMARY_REQUIRED = {
    "setting", "model_name", "target_exam", "mean_mae", "std_mae",
    "n_valid_rounds",
}


def add_check(records, setting, model, check, passed, details=""):
    records.append({
        "setting": setting,
        "model": model,
        "check": check,
        "status": "PASS" if bool(passed) else "FAIL",
        "details": str(details),
    })


def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    adjusted = np.empty_like(p)
    running = 0.0
    m = len(p)
    for rank, index in enumerate(order):
        candidate = (m - rank) * p[index]
        running = max(running, candidate)
        adjusted[index] = min(1.0, running)
    return adjusted


def finite_frame(df, columns):
    values = df[list(columns)].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    return bool(np.isfinite(values).all())


In [ ]:
# CELL A4 — load and validate each of the 12 source runs
validation_records = []
prediction_data = {}
round_data_parts = []
summary_data = {}

for (model, setting), source_dir in SOURCE_DIRS.items():
    pred_path = source_dir / "predictions.csv"
    round_path = source_dir / "per_round.csv"
    summary_path = source_dir / "summary.csv"

    for label, path in [
        ("predictions.csv exists", pred_path),
        ("per_round.csv exists", round_path),
        ("summary.csv exists", summary_path),
    ]:
        add_check(validation_records, setting, model, label, path.is_file(), path)

    if not all(path.is_file() for path in [pred_path, round_path, summary_path]):
        continue

    pred = pd.read_csv(pred_path)
    per_round = pd.read_csv(round_path)
    summary = pd.read_csv(summary_path)

    add_check(validation_records, setting, model, "prediction required columns", PRED_REQUIRED.issubset(pred.columns), sorted(PRED_REQUIRED - set(pred.columns)))
    add_check(validation_records, setting, model, "per_round required columns", ROUND_REQUIRED.issubset(per_round.columns), sorted(ROUND_REQUIRED - set(per_round.columns)))
    add_check(validation_records, setting, model, "summary required columns", SUMMARY_REQUIRED.issubset(summary.columns), sorted(SUMMARY_REQUIRED - set(summary.columns)))

    if not (PRED_REQUIRED.issubset(pred.columns) and ROUND_REQUIRED.issubset(per_round.columns) and SUMMARY_REQUIRED.issubset(summary.columns)):
        continue

    add_check(validation_records, setting, model, "prediction values finite", finite_frame(pred, ["y_true", "y_pred", "abs_error"]), f"rows={len(pred)}")
    add_check(validation_records, setting, model, "round MAEs finite", finite_frame(per_round, ["mae"]), f"rows={len(per_round)}")
    add_check(validation_records, setting, model, "no duplicate prediction keys", not pred.duplicated(["target_exam", "round", "user_id", "item_id"]).any())
    add_check(validation_records, setting, model, "no duplicate round keys", not per_round.duplicated(["target_exam", "round"]).any())
    add_check(validation_records, setting, model, "prediction targets are Mid/Final", set(pred["target_exam"]) == {"Mid", "Final"}, sorted(pred["target_exam"].unique()))
    add_check(validation_records, setting, model, "round targets are Mid/Final/Avg", set(per_round["target_exam"]) == set(TARGETS), sorted(per_round["target_exam"].unique()))

    complete_rounds = all(
        set(group["round"].astype(int)) == EXPECTED_ROUNDS
        for _, group in per_round.groupby("target_exam")
    )
    add_check(validation_records, setting, model, "rounds 1-30 complete for every target", complete_rounds)

    abs_error_ok = np.allclose(
        pred["abs_error"].to_numpy(dtype=float),
        np.abs(pred["y_true"].to_numpy(dtype=float) - pred["y_pred"].to_numpy(dtype=float)),
        rtol=1e-12,
        atol=1e-12,
    )
    add_check(validation_records, setting, model, "abs_error recomputes from y_true/y_pred", abs_error_ok)

    recomputed_exam = (
        pred.groupby(["target_exam", "round"], as_index=False)["abs_error"]
        .mean()
        .rename(columns={"abs_error": "mae_recomputed"})
    )
    saved_exam = per_round[per_round["target_exam"].isin(["Mid", "Final"])][["target_exam", "round", "mae"]]
    exam_check = saved_exam.merge(recomputed_exam, on=["target_exam", "round"], validate="one_to_one")
    add_check(
        validation_records, setting, model,
        "Mid/Final per-round MAE recomputes",
        len(exam_check) == 60 and np.allclose(exam_check["mae"], exam_check["mae_recomputed"], rtol=1e-12, atol=1e-12),
    )

    recomputed_avg = (
        pred.groupby("round", as_index=False)["abs_error"]
        .mean()
        .rename(columns={"abs_error": "mae_recomputed"})
    )
    saved_avg = per_round[per_round["target_exam"].eq("Avg")][["round", "mae"]]
    avg_check = saved_avg.merge(recomputed_avg, on="round", validate="one_to_one")
    add_check(
        validation_records, setting, model,
        "Avg per-round MAE recomputes from pooled Mid+Final errors",
        len(avg_check) == 30 and np.allclose(avg_check["mae"], avg_check["mae_recomputed"], rtol=1e-12, atol=1e-12),
    )

    recomputed_summary = (
        per_round.groupby("target_exam", as_index=False)["mae"]
        .agg(mean_recomputed="mean", std_recomputed="std", n_recomputed="size")
    )
    summary_check = summary.merge(recomputed_summary, on="target_exam", validate="one_to_one")
    summary_ok = (
        set(summary_check["target_exam"]) == set(TARGETS)
        and np.allclose(summary_check["mean_mae"], summary_check["mean_recomputed"], rtol=1e-12, atol=1e-12)
        and np.allclose(summary_check["std_mae"], summary_check["std_recomputed"], rtol=1e-12, atol=1e-12)
        and (summary_check["n_valid_rounds"].astype(int) == summary_check["n_recomputed"].astype(int)).all()
    )
    add_check(validation_records, setting, model, "summary mean/SD/n recomputes", summary_ok)

    pred = pred.copy()
    pred["canonical_model"] = model
    prediction_data[(model, setting)] = pred

    per_round = per_round[["target_exam", "round", "mae"]].copy()
    per_round.insert(0, "model", model)
    per_round.insert(0, "setting", setting)
    round_data_parts.append(per_round)
    summary_data[(model, setting)] = summary.copy()

validation_report = pd.DataFrame(validation_records)
round_level_mae = pd.concat(round_data_parts, ignore_index=True)

initial_failures = validation_report[validation_report["status"].eq("FAIL")]
print(f"Initial validation checks: {len(validation_report)}; failures: {len(initial_failures)}")
if len(initial_failures):
    display(initial_failures)
    raise RuntimeError("Input validation failed. Statistical tests were not run.")


In [ ]:
# CELL A5 — verify exact held-out observation alignment against G-NCDM-poly
# Alignment is required within each setting × target × round.
alignment_records = []

for setting in SETTINGS:
    for target in ["Mid", "Final"]:
        reference = prediction_data[(REFERENCE_MODEL, setting)]
        reference = reference[reference["target_exam"].eq(target)].copy()

        for candidate in CANDIDATE_MODELS:
            model_data = prediction_data[(candidate, setting)]
            model_data = model_data[model_data["target_exam"].eq(target)].copy()

            for round_id in sorted(EXPECTED_ROUNDS):
                left = model_data[model_data["round"].eq(round_id)][["user_id", "item_id", "y_true"]].copy()
                right = reference[reference["round"].eq(round_id)][["user_id", "item_id", "y_true"]].copy()

                left["user_key"] = left["user_id"].astype(str)
                left["item_key"] = left["item_id"].astype(str)
                right["user_key"] = right["user_id"].astype(str)
                right["item_key"] = right["item_id"].astype(str)

                left = left.sort_values(["user_key", "item_key"]).reset_index(drop=True)
                right = right.sort_values(["user_key", "item_key"]).reset_index(drop=True)

                same_keys = left[["user_key", "item_key"]].equals(right[["user_key", "item_key"]])
                same_truth = same_keys and np.allclose(left["y_true"], right["y_true"], rtol=0, atol=0)

                alignment_records.append({
                    "setting": setting,
                    "target_exam": target,
                    "round": round_id,
                    "model": candidate,
                    "reference_model": REFERENCE_MODEL,
                    "n_model": len(left),
                    "n_reference": len(right),
                    "same_observation_keys": bool(same_keys),
                    "same_y_true": bool(same_truth),
                    "status": "PASS" if same_keys and same_truth else "FAIL",
                })

alignment_report = pd.DataFrame(alignment_records)
alignment_summary = (
    alignment_report.groupby(["setting", "target_exam", "model", "reference_model"], as_index=False)
    .agg(
        rounds_checked=("round", "size"),
        rounds_same_keys=("same_observation_keys", "sum"),
        rounds_same_y_true=("same_y_true", "sum"),
        all_pass=("status", lambda x: bool((x == "PASS").all())),
    )
)

print(f"Alignment checks: {len(alignment_report)}; failures: {(alignment_report['status'] == 'FAIL').sum()}")
display(alignment_summary)

if not alignment_summary["all_pass"].all():
    display(alignment_report[alignment_report["status"].eq("FAIL")].head(50))
    raise RuntimeError("Observation alignment failed. Statistical tests were not run.")


In [ ]:
# CELL A6 — paired two-tailed t-tests: candidate MAE != G-NCDM-poly MAE
TEST_DESCRIPTION = "paired two-tailed t-test"
ALTERNATIVE_SUMMARY = "candidate mean MAE != G-NCDM-poly mean MAE"
TITLE = "RQ3 paired two-tailed t-test summary"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "paired_two_tailed"
RESULT_FILENAME = "rq3_paired_two_tailed_results.csv"
SUMMARY_FILENAME = "rq3_paired_two_tailed_summary.txt"

result_records = []

for setting in SETTINGS:
    for target in TARGETS:
        reference = round_level_mae[
            round_level_mae["setting"].eq(setting)
            & round_level_mae["model"].eq(REFERENCE_MODEL)
            & round_level_mae["target_exam"].eq(target)
        ][["round", "mae"]].rename(columns={"mae": "mae_reference"})

        for candidate in CANDIDATE_MODELS:
            model_values = round_level_mae[
                round_level_mae["setting"].eq(setting)
                & round_level_mae["model"].eq(candidate)
                & round_level_mae["target_exam"].eq(target)
            ][["round", "mae"]].rename(columns={"mae": "mae_model"})

            paired = model_values.merge(reference, on="round", how="inner", validate="one_to_one").sort_values("round")
            if set(paired["round"].astype(int)) != EXPECTED_ROUNDS or len(paired) != 30:
                raise RuntimeError(f"Incomplete pairing: {setting}, {target}, {candidate}")

            difference = paired["mae_model"] - paired["mae_reference"]
            test = stats.ttest_rel(
                paired["mae_model"],
                paired["mae_reference"],
                alternative="two-sided",
            )

            mean_model = float(paired["mae_model"].mean())
            mean_reference = float(paired["mae_reference"].mean())
            mean_difference = float(difference.mean())
            sd_difference = float(difference.std(ddof=1))
            standard_error = sd_difference / math.sqrt(len(difference))
            p_value = float(test.pvalue)
            significant = bool(p_value < ALPHA)
            direction = "lower" if mean_difference < 0 else "higher" if mean_difference > 0 else "equal"

            conclusion = (
                f"{candidate} differs significantly from {REFERENCE_MODEL} and has {direction} mean MAE."
                if significant
                else f"No significant difference between {candidate} and {REFERENCE_MODEL}."
            )

            result_records.append({
                "setting": setting,
                "target_exam": target,
                "model": candidate,
                "reference_model": REFERENCE_MODEL,
                "n_pairs": len(paired),
                "n_model": len(paired),
                "n_reference": len(paired),
                "mean_model": mean_model,
                "sd_model": float(paired["mae_model"].std(ddof=1)),
                "mean_reference": mean_reference,
                "sd_reference": float(paired["mae_reference"].std(ddof=1)),
                "mean_difference": mean_difference,
                "direction_vs_reference": direction,
                "sd_difference": sd_difference,
                "standard_error": standard_error,
                "t_statistic": float(test.statistic),
                "degrees_of_freedom": len(paired) - 1,
                "alternative": "two-sided",
                "test_type": "paired_two_tailed_ttest",
                "p_value_raw": p_value,
                "significant_raw_0_05": significant,
                "conclusion": conclusion,
            })

results = pd.DataFrame(result_records)
results["p_value_holm"] = holm_adjust(results["p_value_raw"])
results["significant_holm_0_05"] = results["p_value_holm"] < ALPHA

print(f"Completed {len(results)} paired two-tailed comparisons.")


In [ ]:
# CELL A7 — save validation, round-level inputs, results, and a text summary
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

source_manifest.to_csv(OUTPUT_DIR / "source_manifest.csv", index=False)
validation_report.to_csv(OUTPUT_DIR / "validation_report.csv", index=False)
alignment_report.to_csv(OUTPUT_DIR / "alignment_report.csv", index=False)
round_level_mae.to_csv(OUTPUT_DIR / "round_level_mae.csv", index=False)
results.to_csv(OUTPUT_DIR / RESULT_FILENAME, index=False)

summary_lines = [
    TITLE,
    "",
    f"Test: {TEST_DESCRIPTION}",
    f"Alternative: {ALTERNATIVE_SUMMARY}",
    "Settings are tested separately; Mid, Final, and Avg are tested separately.",
    "Raw p-values are retained; Holm-adjusted p-values are separate columns.",
    f"Input validation: {len(validation_report)} checks, {(validation_report['status'] == 'FAIL').sum()} failures.",
    f"Observation alignment: {len(alignment_report)} checks, {(alignment_report['status'] == 'FAIL').sum()} failures.",
    "",
]

for _, row in results.iterrows():
    summary_lines.append(
        f"{row['setting']} | {row['target_exam']} | {row['model']} vs {row['reference_model']} | "
        f"mean={row['mean_model']:.9f} vs {row['mean_reference']:.9f}, "
        f"difference={row['mean_difference']:.9f}, t={row['t_statistic']:.9f}, "
        f"p_raw={row['p_value_raw']:.12g}, p_Holm={row['p_value_holm']:.12g} | "
        f"{row['conclusion']}"
    )

(OUTPUT_DIR / SUMMARY_FILENAME).write_text("\n".join(summary_lines) + "\n", encoding="utf-8")

print("Saved outputs to:", OUTPUT_DIR)
print("Files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)


In [ ]:
# CELL B1 — display validation reports
display(validation_report.groupby(["status"]).size().rename("count").reset_index())
display(validation_report)
display(alignment_summary)


In [ ]:
# CELL B2 — display all statistical-test results
result_view = results[[
    "setting", "target_exam", "model", "reference_model",
    "n_model", "n_reference", "mean_model", "mean_reference",
    "mean_difference", "t_statistic", "degrees_of_freedom",
    "alternative", "p_value_raw", "significant_raw_0_05",
    "p_value_holm", "significant_holm_0_05", "conclusion",
]].copy()
display(result_view)


In [ ]:
# CELL B3 — compact tables
# †↓ = significant two-tailed difference and lower than G-NCDM-poly
# †↑ = significant two-tailed difference and higher than G-NCDM-poly
for setting in SETTINGS:
    means = (
        round_level_mae[round_level_mae["setting"].eq(setting)]
        .groupby(["model", "target_exam"])["mae"]
        .mean()
        .unstack("target_exam")
    )
    order = ["NCDM-poly", "G-NCDM-poly", "GLLMCDM w/o attempt", "GLLMCDM"]
    compact = pd.DataFrame(index=order, columns=TARGETS)
    for model in order:
        for target in TARGETS:
            compact.loc[model, target] = f"{float(means.loc[model, target]):.4f}"

    subset = results[results["setting"].eq(setting)].set_index(["model", "target_exam"])
    for candidate in CANDIDATE_MODELS:
        for target in TARGETS:
            row = subset.loc[(candidate, target)]
            if bool(row["significant_raw_0_05"]):
                arrow = "↓" if row["mean_difference"] < 0 else "↑"
                compact.loc[candidate, target] += f" †{arrow}"

    print(f"{setting}: † = significant paired two-tailed raw p<0.05; arrow gives direction relative to G-NCDM-poly")
    display(compact)


In [ ]:
# CELL B4 — re-read saved outputs and verify row counts
saved_results = pd.read_csv(OUTPUT_DIR / RESULT_FILENAME)
saved_validation = pd.read_csv(OUTPUT_DIR / "validation_report.csv")
saved_alignment = pd.read_csv(OUTPUT_DIR / "alignment_report.csv")

assert len(saved_results) == 27
assert (saved_validation["status"] == "PASS").all()
assert (saved_alignment["status"] == "PASS").all()
assert np.allclose(saved_results["p_value_raw"], results["p_value_raw"], rtol=1e-12, atol=1e-15)

print("Output verification passed.")
print(f"Results: {len(saved_results)} comparisons")
print(f"Validation: {len(saved_validation)} PASS")
print(f"Alignment: {len(saved_alignment)} PASS")
